In [16]:
import pandas as pd
from sqlalchemy import create_engine

db_user = 'postgres'          
db_password = '03022025' 
db_host = 'localhost'         
db_port = '5432'              
db_name = 'cookie_cats'     

engine = create_engine(f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')

query = """
SELECT * 
FROM cookie_cats 
WHERE sum_gamerounds < 49000;
"""

df = pd.read_sql(query, engine)

df.head(10)

,userid,version_a_b,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True
5,540,gate_40,187,True,True
6,1066,gate_30,0,False,False
7,1444,gate_40,2,False,False
8,1574,gate_40,108,True,True
9,1587,gate_40,153,True,False


In [17]:
# Goal 1: Create a contingency table for Day 1 Retention to prepare for statistical testing.

contingency_table_1=pd.crosstab(
    index=df['version_a_b'],
    columns=df['retention_1']
)

print("Contingency Table - Day 1 Retention:")
print(contingency_table_1)

Contingency Table - Day 1 Retention:
retention_1  False  True 
version_a_b              
gate_30      24665  20034
gate_40      25370  20119


In [ ]:
# Goal 1: Create a contingency table for Day 7 Retention to prepare for statistical testing.
contingency_table_7 = pd.crosstab(
    index=df['version_a_b'],
    columns=df['retention_7']
)
print("contingency table - Day 7 Retention: ")
print(contingency_table_7)

contingency table - Day 7 Retention: 
retention_7  False  True 
version_a_b              
gate_30      36198   8501
gate_40      37210   8279


In [19]:
# Goal 2: Perform Chi-Square test to determine if the difference in Day 1 retention is statistically significant.
from scipy.stats import chi2_contingency
chi2_stats, p_value, dof, expected = chi2_contingency(contingency_table_1)
alpha=0.05
print (f"p_value: {p_value:.5f}")

if p_value < alpha:
    print("Decision: The result is Statistically Significant")
else:
    print("Decision: The result is NOT Statistically Significant")
 

p_value: 0.07501
Decision: The result is NOT Statistically Significant


In [21]:
# Goal 2 (Day 7): Perform Chi-Square test for Day 7 retention
from scipy.stats import chi2_contingency
chi2_stats, p_value, dof, expected = chi2_contingency(contingency_table_7)
alpha = 0.05
print(f"p_value: {p_value:.5f}")
if p_value < alpha:
    print("Decision: The result is Statistically Significant")
else:
    print("Decision: The result is NOT Statistically Significant")

p_value: 0.00164
Decision: The result is Statistically Significant


In [22]:
import pandas as pd

boot_1d = []
boot_7d = []

for i in range(500):
    sample = df.sample(frac=1, replace= True)
    mean_1d = sample.groupby('version_a_b')['retention_1'].mean()
    boot_1d.append(mean_1d)
    mean_7d = sample.groupby('version_a_b')['retention_7'].mean()
    boot_7d.append(mean_7d)

boot_1d_df = pd.DataFrame(boot_1d)
boot_7d_df = pd.DataFrame(boot_7d)

boot_1d_df['diff'] = (boot_1d_df['gate_30'] - boot_1d_df['gate_40']) / boot_1d_df['gate_40'] * 100
boot_7d_df['diff'] = (boot_7d_df['gate_30'] - boot_7d_df['gate_40']) / boot_7d_df['gate_40'] * 100

print("Bootstrapping results for Day 1 (first 5 simulations):")
print(boot_1d_df.head())

print("Bootstrapping results for Day 7 (first 5 simulations):")
print(boot_7d_df.head())

Bootstrapping results for Day 1 (first 5 simulations):
version_a_b   gate_30   gate_40      diff
retention_1  0.446439  0.441401  1.141370
retention_1  0.442366  0.440824  0.349786
retention_1  0.443497  0.443188  0.069667
retention_1  0.451561  0.443471  1.824264
retention_1  0.449897  0.442448  1.683584
Bootstrapping results for Day 7 (first 5 simulations):
version_a_b   gate_30   gate_40      diff
retention_7  0.191210  0.183980  3.929994
retention_7  0.191377  0.180796  5.852226
retention_7  0.189272  0.181470  4.299419
retention_7  0.193398  0.181820  6.367802
retention_7  0.190761  0.181374  5.175726


In [23]:
df.to_csv('cookie_cats_cleaned.csv', index=False)